In [ ]:
# Import necessary libraries
from pathlib import Path
import json
import pandas as pd
import numpy as np
import re

In [ ]:
# File paths
INPUT_DIR = Path(r"data evaluation")
LONG_FILE = INPUT_DIR / "deepeval_scores.long.jsonl"
WIDE_FILE = INPUT_DIR / "deepeval_scores.wide.csv"
OUT_FILE  = INPUT_DIR / "llm_eval_summary_by_dimension.csv"

In [ ]:
# Order to display dimensions
DIM_ORDER = ["fluency", "naturalness", "clarity", "structure"]


def load_from_long(long_path: Path) -> pd.DataFrame:
    """
    Load rows from deepeval_scores.long.jsonl.
    Expected columns: dialogue_id, dimension, expected, entropy, ...

    """
    records = []
    with long_path.open("r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            obj = json.loads(line)
            records.append({
                "dialogue_id": obj.get("dialogue_id"),
                "dimension":   obj.get("dimension"),
                "expected":    obj.get("expected"),
                "entropy":     obj.get("entropy"),
            })
    df = pd.DataFrame(records)
    # Ensure numeric
    df["expected"] = pd.to_numeric(df["expected"], errors="coerce")
    df["entropy"]  = pd.to_numeric(df["entropy"],  errors="coerce")
    return df


def load_from_wide(wide_path: Path) -> pd.DataFrame:
    """
    Load rows from deepeval_scores.wide.csv and reshape to long format.
    Columns look like: Naturalness_score, Naturalness_entropy, etc.

    """
    wide = pd.read_csv(wide_path)
    # Split score columns and entropy columns
    score_cols   = [c for c in wide.columns if c.endswith("_score")]
    entropy_cols = [c for c in wide.columns if c.endswith("_entropy")]

    # Melt scores
    scores_long = wide.melt(
        id_vars=["dialogue_id"] if "dialogue_id" in wide.columns else None,
        value_vars=score_cols,
        var_name="dimension_score",
        value_name="expected"
    )
    # Get the dimension name (before _score)
    scores_long["dimension"] = scores_long["dimension_score"].str.replace("_score", "", regex=False).str.lower()

    # Melt entropy
    entropy_long = wide.melt(
        id_vars=["dialogue_id"] if "dialogue_id" in wide.columns else None,
        value_vars=entropy_cols,
        var_name="dimension_entropy",
        value_name="entropy"
    )
    entropy_long["dimension"] = entropy_long["dimension_entropy"].str.replace("_entropy", "", regex=False).str.lower()

    # Merge on dialogue_id + dimension
    how_merge = "inner" if "dialogue_id" in scores_long.columns else "left"
    on_cols = ["dialogue_id", "dimension"] if "dialogue_id" in scores_long.columns else ["dimension"]
    df = pd.merge(scores_long[on_cols + ["expected"]],
                  entropy_long[on_cols + ["entropy"]],
                  on=on_cols, how=how_merge)

    # Ensure numeric
    df["expected"] = pd.to_numeric(df["expected"], errors="coerce")
    df["entropy"]  = pd.to_numeric(df["entropy"],  errors="coerce")
    return df


def load_scores(long_path: Path, wide_path: Path) -> pd.DataFrame:
    """
    Load scores from either the long JSONL or wide CSV file.
    
    """
    if long_path.exists():
        print(f"Reading: {long_path}")
        return load_from_long(long_path)
    elif wide_path.exists():
        print(f"Reading: {wide_path}")
        return load_from_wide(wide_path)
    else:
        raise FileNotFoundError(f"Neither file found:\n - {long_path}\n - {wide_path}")


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    """
    Summarize scores by dimension.
    
    """
    # Normalize dimension names
    df["dimension"] = df["dimension"].str.lower().str.strip()

    # Group statistics
    grp = df.groupby("dimension", as_index=False).agg(
        n=("expected", "count"),
        expected_mean=("expected", "mean"),
        expected_std =("expected", "std"),
        entropy_mean =("entropy",  "mean"),
        entropy_std  =("entropy",  "std"),
    )

    # Order rows
    order = [d for d in DIM_ORDER if d in set(grp["dimension"])]
    grp = grp.set_index("dimension").loc[order].reset_index()

    # Pretty names + rounding
    grp["Dimension"] = grp["dimension"].str.capitalize()
    grp = grp.drop(columns=["dimension"]).rename(columns={
        "n": "N",
        "expected_mean": "Avg. Expected",
        "expected_std":  "SD (Expected)",
        "entropy_mean":  "Avg. Entropy",
        "entropy_std":   "SD (Entropy)"
    })
    grp = grp[["Dimension", "N", "Avg. Expected", "SD (Expected)", "Avg. Entropy", "SD (Entropy)"]]
    grp = grp.round({"Avg. Expected": 3, "SD (Expected)": 3, "Avg. Entropy": 3, "SD (Entropy)": 3})
    return grp


def latex_table(df: pd.DataFrame) -> str:
    """
    Convert summary DataFrame to LaTeX table string.
    
    """
    # Compact LaTeX tabular (no index)
    return df.to_latex(index=False, escape=True)

# Main execution
if __name__ == "__main__":
    df = load_scores(LONG_FILE, WIDE_FILE)
    summary = summarize(df)

    # Save CSV + print LaTeX
    summary.to_csv(OUT_FILE, index=False)
    print(f"Saved summary CSV → {OUT_FILE}")
    print("\nLaTeX table:\n")
    print(latex_table(summary))

In [ ]:
# File paths for dialogues and scores
dialogues_file = Path("evaluation_dialogues.jsonl")  # persona + dialogue text
scores_long_file = Path("deepeval_scores.long.jsonl")  # expected + entropy per dimension
scores_wide_file = Path("deepeval_scores.wide.csv")   # already wide format (backup)



# Load dialogues with persona and first speaker
def extract_persona_name(dialogue: str) -> str:
    """
    Extract persona name from a dialogue.
    Assumes names appear in agent addressing (e.g., 'Jan, ...', 'Hello Aicha, ...').
    If no match, returns 'Unknown'.

    """
    # Look for capitalized name (first word after Agent: or Patient:)
    match = re.search(r"\b(Agent|Patient):\s*([A-Z][a-z]+)", dialogue)
    if match:
        return match.group(2)
    return "Unknown"

def extract_first_speaker(dialogue: str) -> str:
    """
    Detect whether the first line starts with Agent or Patient.

    """
    first_line = dialogue.split("\n")[0].strip()
    if first_line.startswith("Agent:"):
        return "Agent"
    elif first_line.startswith("Patient:"):
        return "Patient"
    return "Unknown"

dialogue_meta = []
with open(dialogues_file, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        did = obj["dialogue_id"]
        dlg = obj["dialogue"]
        persona = extract_persona_name(dlg)
        starter = extract_first_speaker(dlg)
        dialogue_meta.append({"dialogue_id": did, "persona": persona, "starter": starter})

df_meta = pd.DataFrame(dialogue_meta)


# Load evaluation scores (from long JSONL, more detailed than wide)
records = []
with open(scores_long_file, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        records.append({
            "dialogue_id": obj.get("dialogue_id"),
            "dimension":   obj.get("dimension"),
            "expected":    obj.get("expected"),
            "entropy":     obj.get("entropy")
        })

df_scores = pd.DataFrame(records)

# Merge dialogues with scores
df = df_scores.merge(df_meta, on="dialogue_id", how="left")

# Summary per persona
persona_summary = df.groupby(["persona", "dimension"]).agg(
    n=("expected", "count"),
    expected_mean=("expected", "mean"),
    expected_std=("expected", "std"),
    entropy_mean=("entropy", "mean"),
    entropy_std=("entropy", "std"),
).reset_index()

# Pivot to wide table
persona_wide = persona_summary.pivot(
    index="persona", columns="dimension",
    values=["expected_mean", "expected_std", "entropy_mean", "entropy_std"]
)
persona_wide = persona_wide.round(3)

# Summary per starter (Agent vs Patient)
starter_summary = df.groupby(["starter", "dimension"]).agg(
    n=("expected", "count"),
    expected_mean=("expected", "mean"),
    expected_std=("expected", "std"),
    entropy_mean=("entropy", "mean"),
    entropy_std=("entropy", "std"),
).reset_index()

starter_wide = starter_summary.pivot(
    index="starter", columns="dimension",
    values=["expected_mean", "expected_std", "entropy_mean", "entropy_std"]
)
starter_wide = starter_wide.round(3)

# Save outputs
persona_wide.to_csv("persona_eval_summary.csv")
starter_wide.to_csv("starter_eval_summary.csv")

print("Saved persona_eval_summary.csv and starter_eval_summary.csv")
print("\nPersona summary preview:")
print(persona_wide.head())

print("\nStarter summary preview:")
print(starter_wide)

Saved persona_eval_summary.csv and starter_eval_summary.csv

Persona summary preview:
           expected_mean                               expected_std          \
dimension        clarity fluency naturalness structure      clarity fluency   
persona                                                                       
Abdullah           4.456   4.733       4.478     4.111        0.053   0.132   
Absolutely         4.500   4.580       4.500     4.200        0.000   0.130   
Aicha              4.450   4.650       4.475     4.050        0.053   0.160   
Ali                4.488   4.625       4.488     4.212        0.035   0.149   
Anxiety            4.500   4.800       4.500     4.200          NaN     NaN   

                                 entropy_mean                                \
dimension  naturalness structure      clarity fluency naturalness structure   
persona                                                                       
Abdullah         0.044     0.127        0.91